In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Loading synthetic...")
synthetic = pd.read_csv('../data/synthetic/synthetic_data.csv')
print(f"✅ Synthetic data: {synthetic.shape}")

Loading synthetic...
✅ Synthetic data: (50000, 22)


In [2]:
print("Loading hypertension...")
hypertension = pd.read_csv('../data/processed/hypertension_features.csv')
print(f"✅ Hypertension features: {hypertension.shape}")

Loading hypertension...
✅ Hypertension features: (118502, 16)


In [3]:
print("Loading diabetes...")
diabetes = pd.read_csv('../data/processed/diabetes_features.csv')
print(f"✅ Diabetes features: {diabetes.shape}")

Loading diabetes...
✅ Diabetes features: (304448, 17)


In [4]:
health_cols = [
    'age', 'bmi',
    'avg_systolic', 'avg_diastolic', 'bp_variability',
    'avg_glucose', 'sugar_variability',
    'avg_pulse', 'adherence_rate', 'missed_doses',
    'has_chronic_condition', 'allergies_count',
    'physical_activity_score', 'diet_quality_score',
    'stress_score', 'sleep_score',
    'health_score'
]

synthetic_hs = synthetic[health_cols].copy()
print(f"✅ Synthetic ready: {synthetic_hs.shape}")

✅ Synthetic ready: (50000, 17)


In [5]:
def calculate_health_score(df):
    score = (
        100
        - (df['avg_systolic'] - 110).clip(0, 90) * 0.3
        - (df['avg_glucose'] - 90).clip(0, 260) * 0.15
        - (df['bmi'] - 22).clip(0, 28) * 0.8
        - df['missed_doses'] * 1.2
        - df['stress_score'] * 0.8
        - df['bp_variability'] * 0.4
        - df['sugar_variability'] * 0.2
        + df['physical_activity_score'] * 1.5
        + df['diet_quality_score'] * 1.0
        + df['sleep_score'] * 0.8
        + df['adherence_rate'] * 10
        + np.random.normal(0, 2, len(df))
    ).clip(10, 100).round(1)
    return score

diabetes_hs = diabetes.copy()
diabetes_hs['health_score'] = calculate_health_score(diabetes_hs)

if 'allergies_count' not in diabetes_hs.columns:
    diabetes_hs['allergies_count'] = np.random.choice(
        [0,1,2,3], len(diabetes_hs), p=[0.5,0.25,0.15,0.10]
    )

diabetes_hs = diabetes_hs[health_cols]
print(f"✅ Diabetes with health score: {diabetes_hs.shape}")
print(f"   Avg health score: {diabetes_hs['health_score'].mean():.1f}")

✅ Diabetes with health score: (304448, 17)
   Avg health score: 93.3


In [6]:
hypertension_hs = hypertension.copy()
hypertension_hs['health_score'] = calculate_health_score(hypertension_hs)

if 'allergies_count' not in hypertension_hs.columns:
    hypertension_hs['allergies_count'] = np.random.choice(
        [0,1,2,3], len(hypertension_hs), p=[0.5,0.25,0.15,0.10]
    )

hypertension_hs = hypertension_hs[health_cols]
print(f"✅ Hypertension with health score: {hypertension_hs.shape}")
print(f"   Avg health score: {hypertension_hs['health_score'].mean():.1f}")

✅ Hypertension with health score: (118502, 17)
   Avg health score: 94.9


In [7]:
combined = pd.concat([
    diabetes_hs,
    hypertension_hs,
    synthetic_hs
], ignore_index=True)

combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
combined = combined.fillna(combined.median(numeric_only=True))

print(f"✅ Combined: {combined.shape}")
print(f"\nHealth Score Distribution:")
print(f"   Poor  (10-40):   {(combined['health_score'] < 40).sum():,}")
print(f"   Fair  (40-60):   {((combined['health_score'] >= 40) & (combined['health_score'] < 60)).sum():,}")
print(f"   Good  (60-80):   {((combined['health_score'] >= 60) & (combined['health_score'] < 80)).sum():,}")
print(f"   Excellent (80+): {(combined['health_score'] >= 80).sum():,}")
print(f"\nNull values: {combined.isnull().sum().sum()}")

✅ Combined: (472950, 17)

Health Score Distribution:
   Poor  (10-40):   382
   Fair  (40-60):   6,591
   Good  (60-80):   42,006
   Excellent (80+): 423,971

Null values: 0


In [8]:
output_path = '../data/processed/health_score_features.csv'
combined.to_csv(output_path, index=False)

print(f"✅ Health score features saved!")
print(f"   Path: {output_path}")
print(f"   Rows: {len(combined):,}")
print(f"   Size: {os.path.getsize(output_path)/1024/1024:.1f} MB")

✅ Health score features saved!
   Path: ../data/processed/health_score_features.csv
   Rows: 472,950
   Size: 33.3 MB
